In [1]:
import sys
sys.path.append('../src')
from preprocessing import load_and_optimize_data, create_features
from model import prepare_and_split, train_xgboost_baseline

# 1. Cargar datos optimizados
df = load_and_optimize_data('../data/raw/credit_card_transactions.csv')

# 2. Generar nuevas variables
df_features = create_features(df)

# 3. Dividir datos
X_train, X_test, y_train, y_test = prepare_and_split(df_features)

# 4. Entrenar y evaluar
model = train_xgboost_baseline(X_train, y_train, X_test, y_test)

Cargando datos desde: ../data/raw/credit_card_transactions.csv...
Uso de memoria inicial del DataFrame: 95.37 MB
Uso de memoria final tras optimización: 26.97 MB
Reducción de memoria: 71.7%
Iniciando Feature Engineering...
Feature Engineering completado. Nuevas variables añadidas.
Preparando datos para modelado...
Dimensiones Train: (400000, 24)
Dimensiones Test: (100000, 24)
Entrenando modelo XGBoost Baseline...

--- Resultados Baseline ---
PR-AUC Score: 0.0305

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.73      0.84     97664
           1       0.03      0.37      0.06      2336

    accuracy                           0.72    100000
   macro avg       0.51      0.55      0.45    100000
weighted avg       0.96      0.72      0.82    100000



## **1. El Impacto del scale_pos_weight**
Al usar este parámetro, le dijimos a XGBoost: "Equivocarte al clasificar un fraude cuesta 40 veces más que equivocarte con una transacción legítima".
El modelo hizo exactamente lo que le pedimos: se volvió extremadamente paranoico. Ante la menor duda, marca la transacción como fraude.

## **2. Lectura de Métricas**
* **Recall (Clase 1) = 0.37:** Estamos capturando el 37% de todos los fraudes reales. No está mal para un modelo sin optimizar.

* **Recall (Clase 0) = 0.73:** ¡Alerta roja! Esto significa que estamos aprobando solo el 73% de las transacciones legítimas. Es decir, estamos declinando por error el 27% de las compras de clientes honestos. En la vida real, esto colapsaría el centro de atención al cliente de cualquier banco.

* **Precision (Clase 1) = 0.03:** De cada 100 transacciones que nuestro modelo bloquea por sospecha de fraude, solo 3 son realmente fraude. Los otros 97 son Falsos Positivos.

* **PR-AUC = 0.0305:** Esta es nuestra línea base. El objetivo del proyecto será subir este número calibrando el modelo.

# **3. Optimizacion y Ajustes**

El problema radica en que classification_report utiliza un umbral de decisión (threshold) por defecto del 50% (0.5). Es decir, si el modelo dice que hay un 51% de probabilidad de fraude, lo bloquea.

Como forzamos al modelo a ser paranoico con el scale_pos_weight, muchas transacciones legítimas están arrojando, digamos, un 60% de probabilidad de fraude en la cabeza del algoritmo.

Tenemos dos caminos claros para mejorar drásticamente estas métricas y acercarnos a un modelo listo para producción:

* Optimización de Umbral (Threshold Moving): Escribir un pequeño bloque de código para encontrar el punto de corte óptimo (bloquear solo si la probabilidad es > 0.85) para equilibrar la precisión y el recall sin reentrenar el modelo.

* Ajuste de Hiperparámetros (Tuning): El modelo actual es muy básico (max_depth=5, n_estimators=100). Podemos usar validación cruzada para encontrar hiperparámetros que permitan al árbol crear reglas mucho más complejas y precisas.